In [ ]:
import os
import torch
import numpy as np
import pandas as pd
from torch import nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from tqdm import tqdm

In [ ]:
MAX_LEN = 128
BATCH_SIZE = 64
U_EPOCHS = 1
SEED = 42
MODEL_PATH = "/kaggle/input/deberta-v3-base/transformers/default/1/deberta-v3-base"

In [ ]:
import random
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print(f"Number of GPUs available: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")

In [ ]:
print("Loading unlabeled data...")
unlabelled1 = pd.read_csv('/kaggle/input/199kv2-jigsaw/test.csv').iloc[31750:]
unlabelled2 = pd.read_csv('/kaggle/input/jigsaw500k/test.csv')
unlabelled2['row_id'] += len(unlabelled1)

unlabelled3 = pd.read_csv('/kaggle/input/jigsaw500k2/test.csv')
unlabelled3['row_id'] += unlabelled2.row_id.iloc[-1] + 1

unlabelled = pd.concat([unlabelled1, unlabelled2, unlabelled3], axis=0, ignore_index=True)
unlabelled['rule'] = unlabelled['rule'].str.lower().str.strip()
unlabelled['text'] = unlabelled['rule'] + ' [SEP] ' + unlabelled['body']

print(f"Total unlabeled examples: {len(unlabelled)}")
print(f"Label distribution:\n{unlabelled.rule_violation.describe()}")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, use_fast=False)

In [ ]:
class PseudoDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        enc = self.tokenizer(
            text, 
            padding='max_length', 
            truncation=True, 
            max_length=self.max_len, 
            return_tensors="pt"
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.float)
        return item

In [ ]:
class JigsawModel(nn.Module):
    def __init__(self, model_path):
        super().__init__()
        self.base = AutoModel.from_pretrained(model_path)
        self.drop = nn.Dropout(0.15)
        self.out = nn.Linear(self.base.config.hidden_size, 1)

    def forward(self, input_ids, attention_mask):
        outputs = self.base(input_ids=input_ids, attention_mask=attention_mask)
        pooled = outputs.last_hidden_state[:, 0]
        return self.out(self.drop(pooled)).squeeze(1)

In [ ]:
class BinaryFocalLossWithSoftLabels(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction
    
    def forward(self, logits, soft_targets):
        if logits.dim() > 1:
            logits = logits.squeeze(-1)
        if soft_targets.dim() > 1:
            soft_targets = soft_targets.squeeze(-1)
        
        probs = torch.sigmoid(logits)
        bce_loss = -(soft_targets * torch.log(probs + 1e-8) + 
                     (1 - soft_targets) * torch.log(1 - probs + 1e-8))
        p_t = soft_targets * probs + (1 - soft_targets) * (1 - probs)
        focal_weight = (1 - p_t) ** self.gamma
        focal_loss = focal_weight * bce_loss
        
        if self.alpha is not None:
            alpha_weight = soft_targets * self.alpha + (1 - soft_targets) * (1 - self.alpha)
            focal_loss = alpha_weight * focal_loss
        
        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        else:
            return focal_loss


def compute_alpha_from_soft_labels(soft_labels):
    if isinstance(soft_labels, torch.Tensor):
        if soft_labels.dim() > 1:
            soft_labels = soft_labels.squeeze(-1)
        pos_proportion = soft_labels.mean().item()
    else:
        pos_proportion = np.mean(soft_labels)
    alpha = 1 - pos_proportion
    return alpha

In [ ]:
texts = unlabelled['text'].tolist()
soft_labels = unlabelled['rule_violation'].tolist()

alpha = compute_alpha_from_soft_labels(soft_labels)
print(f"Computed alpha for positive class weighting: {alpha:.4f}")

dataset = PseudoDataset(texts, soft_labels, tokenizer, MAX_LEN)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)

print(f"Dataset size: {len(dataset)}")
print(f"Number of batches: {len(loader)}")
print(f"Effective batch size with 2 GPUs: {BATCH_SIZE}")

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Primary device: {device}")

model = JigsawModel(MODEL_PATH)

for name, param in model.named_parameters():
    if name.startswith('base.embeddings'):
        param.requires_grad = False

if torch.cuda.device_count() > 1:
    print(f"Using DataParallel with {torch.cuda.device_count()} GPUs")
    model = nn.DataParallel(model)

model = model.to(device)

criterion = BinaryFocalLossWithSoftLabels(alpha=alpha, gamma=2.0)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5, eps=1e-6)

total_steps = U_EPOCHS * len(loader)
warmup_steps = int(0.1 * total_steps)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

print(f"Total training steps: {total_steps}")
print(f"Warmup steps: {warmup_steps}")

In [ ]:
print(f"Starting pretraining for {U_EPOCHS} epoch(s)...\n")

for epoch in range(U_EPOCHS):
    model.train()
    total_loss = 0
    
    progress_bar = tqdm(loader, desc=f'Epoch {epoch+1}/{U_EPOCHS}')
    for batch_idx, batch in enumerate(progress_bar):
        optimizer.zero_grad()
        
        input_ids = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        
        logits = model(input_ids, mask)
        loss = criterion(logits, labels)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        
        total_loss += loss.item()
        
        if (batch_idx + 1) % 100 == 0:
            avg_loss = total_loss / (batch_idx + 1)
            progress_bar.set_postfix({'loss': f'{avg_loss:.4f}'})
    
    avg_epoch_loss = total_loss / len(loader)
    print(f"Epoch {epoch+1}/{U_EPOCHS} - Average Loss: {avg_epoch_loss:.4f}")

print("\nPretraining complete!")

In [ ]:
if isinstance(model, nn.DataParallel):
    model_to_save = model.module
else:
    model_to_save = model

torch.save(model_to_save.state_dict(), "pretrained_model.bin")
print("Model saved to: pretrained_model.bin")